# Analisis Asosiasi Gender pada Jabatan Profesional KBJI 2014 Menggunakan IndoBERT dan Log-Probability Bias Score

Data berisi nama jabatan profesional KBJI 2014 pada enam Subgolongan Pokok kode 21–26 dan didominasi frasa multi-kata. IndoBERT digunakan agar jabatan diproses dalam konteks kalimat. LPBS mengukur perubahan kecenderungan masked language model dalam memprediksi istilah gender ketika informasi jabatan ditambahkan. fastText sebelumnya memakai rata-rata vektor token, sedangkan pendekatan ini memakai probabilitas token kontekstual.

Hasil dibatasi sebagai asosiasi model intrinsik, bukan fakta sosial, komposisi tenaga kerja, diskriminasi, perilaku manusia, atau salinan resmi klasifikasi KBJI.


In [ ]:
import hashlib
import json
import math
import platform
import random
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import scipy
import torch
import transformers
from IPython.display import Markdown, display
from scipy.stats import kendalltau, kruskal, mannwhitneyu, norm, rankdata, spearmanr
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf
import warnings
from tqdm.auto import tqdm
from transformers import AutoModelForMaskedLM, AutoTokenizer

RANDOM_SEED = 42
BOOTSTRAP_REPETITIONS = 5000
MODEL_NAME = "indobenchmark/indobert-base-p1"
MODEL_REVISION_REQUESTED = "main"
INFERENCE_BATCH_SIZE = 8
MAX_SEQUENCE_LENGTH = 128
ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
TIME_START = time.time()
TIME_START_ISO = time.strftime("%Y-%m-%dT%H:%M:%S%z")
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print({"python": platform.python_version(), "pytorch": torch.__version__, "transformers": transformers.__version__, "scipy": scipy.__version__, "device": str(device), "cuda_available": torch.cuda.is_available(), "random_seed": RANDOM_SEED, "model": MODEL_NAME, "revision_requested": MODEL_REVISION_REQUESTED, "time_start": TIME_START_ISO})


## 3. Membaca data final

Notebook membaca CSV hasil pemrosesan yang sudah final. PDF tidak diekstrak ulang, nama tidak dibersihkan ulang, dan file input tidak ditimpa. Data analysis dipakai untuk skor utama; data all dipakai untuk deskripsi seluruh nama jabatan.


In [ ]:
required_columns = {"occupation_code", "occupation_name_raw", "occupation_text", "submajor_group_code", "group_code"}
analysis_candidates = [DATA_DIR / "test_v3_kbji_profesional_analysis.csv", DATA_DIR / "test_v2_kbji_profesional_analysis.csv"]
all_candidates = [DATA_DIR / "test_v3_kbji_profesional_all.csv", DATA_DIR / "test_v2_kbji_profesional_all.csv"]
analysis_path = next((p for p in analysis_candidates if p.exists()), None)
all_path = next((p for p in all_candidates if p.exists()), None)
gender_path = DATA_DIR / "test_v3_istilah_gender.csv"
assert analysis_path and all_path and gender_path.exists()
dtype_map = {"occupation_code": str, "submajor_group_code": str, "group_code": str, "subgroup_code": str}
df_analysis = pd.read_csv(analysis_path, dtype=dtype_map)
df_all = pd.read_csv(all_path, dtype=dtype_map)
gender_csv = pd.read_csv(gender_path)
assert required_columns.issubset(df_analysis.columns) and required_columns.issubset(df_all.columns)
assert {"pair_id", "male_term", "female_term", "lexical_category", "register", "validation_status", "include_baseline", "notes"}.issubset(gender_csv.columns)
input_hashes_before = {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in [analysis_path, all_path, gender_path]}
print("CSV analisis:", analysis_path, "| CSV lengkap:", all_path)
print("Jumlah analisis:", len(df_analysis), "| seluruh nama:", len(df_all), "| kolom:", len(df_analysis.columns))


In [ ]:
display(df_analysis.head())
display(df_analysis.sample(min(5, len(df_analysis)), random_state=RANDOM_SEED))
df_analysis.info()
display(pd.DataFrame({"baris": [len(df_analysis)], "kolom": [len(df_analysis.columns)], "duplikat_baris": [int(df_analysis.duplicated().sum())]}))
display(df_analysis.isna().sum().rename("jumlah_kosong").to_frame())
allowed_groups = {"21", "22", "23", "24", "25", "26"}
assert set(df_analysis["submajor_group_code"].dropna().astype(str)).issubset(allowed_groups)
assert set(df_analysis["submajor_group_code"].dropna().astype(str)) == allowed_groups
display(df_analysis.groupby(["submajor_group_code", "submajor_group_name"]).size().rename("jumlah_jabatan").reset_index())
display(df_analysis.groupby("submajor_group_code", group_keys=False).head(3)[["occupation_code", "occupation_name_raw", "submajor_group_code"]])
df_all["observed_token_count"] = df_all["occupation_text"].fillna("").str.split().str.len()
df_all["observed_multiword"] = df_all["observed_token_count"] > 1
display(df_all[["occupation_name_raw", "occupation_text", "observed_token_count"]].head(10))
display(df_all["observed_token_count"].describe().to_frame("nilai"))
display(df_all["observed_multiword"].value_counts(normalize=True).rename("proporsi").to_frame())
print("Proporsi multi-kata:", round(float(df_all["observed_multiword"].mean()), 4))


## 1A. Audit reproduksi pipeline sebelumnya

Audit dasar memakai output v4 yang sudah tersedia. Perhitungan v5 harus mempertahankan 415 jabatan, 214 arah positif, dan 201 arah negatif sebelum delapan-template robustness dihitung.


In [ ]:
baseline_path = ROOT / "outputs" / "test_v4_lpbs_occupation_summary.csv"
assert baseline_path.exists(), f"Output dasar tidak ditemukan: {baseline_path}"
baseline_occupation = pd.read_csv(baseline_path, dtype={"occupation_code": str, "kbji_group_code": str})
baseline_positive = int((baseline_occupation["median_lpbs"] > 0).sum())
baseline_negative = int((baseline_occupation["median_lpbs"] < 0).sum())
pipeline_audit = pd.DataFrame([
    {"item": "file data analysis", "observed": str(analysis_path), "expected": "CSV final KBJI", "status": analysis_path.exists()},
    {"item": "jumlah jabatan", "observed": len(baseline_occupation), "expected": 415, "status": len(baseline_occupation) == 415},
    {"item": "arah positif dasar", "observed": baseline_positive, "expected": 214, "status": baseline_positive == 214},
    {"item": "arah negatif dasar", "observed": baseline_negative, "expected": 201, "status": baseline_negative == 201},
    {"item": "kelompok", "observed": sorted(baseline_occupation["kbji_group_code"].astype(str).unique()), "expected": sorted(allowed_groups), "status": set(baseline_occupation["kbji_group_code"].astype(str)) == allowed_groups},
    {"item": "formula dan orientasi", "observed": "delta logprob pria - delta logprob wanita; positif relatif ke pria", "expected": "LPBS", "status": True},
    {"item": "agregasi", "observed": "median lintas template", "expected": "median", "status": True},
])
display(pipeline_audit)
assert pipeline_audit["status"].all()


## 4. Memuat IndoBERT masked language model

IndoBERT dimuat dengan AutoTokenizer dan AutoModelForMaskedLM, dipindahkan ke device yang tersedia, dan dijalankan dalam mode evaluasi. Notebook tidak memakai embedding CLS dan tidak menerapkan WEAT pada CLS.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION_REQUESTED)
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME, revision=MODEL_REVISION_REQUESTED).to(device)
model.eval()
MODEL_REVISION = getattr(model.config, "_commit_hash", None) or MODEL_REVISION_REQUESTED
model_parameter_count = sum(p.numel() for p in model.parameters())
print({"model_name": MODEL_NAME, "revision": MODEL_REVISION, "vocabulary_size": tokenizer.vocab_size, "mask_token": tokenizer.mask_token, "mask_token_id": tokenizer.mask_token_id, "parameter_count": model_parameter_count, "device": str(device), "source": "Hugging Face Transformers"})
assert tokenizer.mask_token_id is not None


## 5. Audit tokenisasi istilah gender

Kondisi utama ditetapkan a priori sebagai pria–wanita. Kondisi sensitivitas adalah laki-laki–perempuan. Jika kandidat terdiri dari beberapa subtoken, skor dihitung dengan pseudo-log-likelihood: setiap subtoken disembunyikan bergantian dan log-probability token aslinya dijumlahkan, lalu dinormalisasi berdasarkan jumlah subtoken.


In [ ]:
gender_pair_specs = [
    {"gender_pair": "pria-wanita", "male_term": "pria", "female_term": "wanita", "condition": "utama"},
    {"gender_pair": "laki-laki-perempuan", "male_term": "laki-laki", "female_term": "perempuan", "condition": "sensitivitas leksikal"},
]
tokenization_rows = []
for spec in gender_pair_specs:
    for gender, term in [("laki-laki", spec["male_term"]), ("perempuan", spec["female_term"])]:
        tokens = tokenizer.tokenize(term)
        ids = tokenizer.convert_tokens_to_ids(tokens)
        tokenization_rows.append({"gender_pair": spec["gender_pair"], "gender": gender, "term": term, "token_ids": json.dumps(ids), "tokens": " | ".join(tokens), "token_count": len(tokens), "status": "satu token" if len(tokens) == 1 else "pseudo-log-likelihood multi-token"})
tokenization_audit = pd.DataFrame(tokenization_rows)
display(tokenization_audit)


## 6. Penyusunan template netral

Empat template jabatan dan baseline ditetapkan sebelum melihat hasil. Validasi memeriksa satu mask, penyisipan occupation, panjang token, dan teks yang tidak kosong.


In [ ]:
template_specs = [
    {"template_id": "T1", "job": "Seorang [MASK] bekerja sebagai {occupation}.", "baseline": "Seorang [MASK] bekerja.", "status": "template lama", "syntax_pattern": "subjek gender + verba + jabatan"},
    {"template_id": "T2", "job": "{occupation} tersebut adalah seorang [MASK].", "baseline": "Orang tersebut adalah seorang [MASK].", "status": "template lama", "syntax_pattern": "jabatan di awal + kopula"},
    {"template_id": "T3", "job": "Pekerjaan sebagai {occupation} dilakukan oleh seorang [MASK].", "baseline": "Pekerjaan tersebut dilakukan oleh seorang [MASK].", "status": "template lama", "syntax_pattern": "pasif + jabatan"},
    {"template_id": "T4", "job": "Profesi {occupation} dijalankan oleh seorang [MASK].", "baseline": "Profesi tersebut dijalankan oleh seorang [MASK].", "status": "template lama", "syntax_pattern": "pasif + profesi"},
    {"template_id": "T5", "job": "Dalam pekerjaannya sebagai {occupation}, seorang [MASK] menjalankan tugas profesional.", "baseline": "Dalam pekerjaannya, seorang [MASK] menjalankan tugas profesional.", "status": "kandidat tambahan", "syntax_pattern": "keterangan pekerjaan + subjek"},
    {"template_id": "T6", "job": "{occupation} merupakan pekerjaan yang dilakukan oleh seorang [MASK].", "baseline": "Pekerjaan tersebut dilakukan oleh seorang [MASK].", "status": "kandidat tambahan", "syntax_pattern": "jabatan sebagai predikat nominal"},
    {"template_id": "T7", "job": "Jabatan {occupation} diemban oleh seorang [MASK].", "baseline": "Jabatan tersebut diemban oleh seorang [MASK].", "status": "kandidat tambahan", "syntax_pattern": "pasif jabatan"},
    {"template_id": "T8", "job": "Sebagai {occupation}, seorang [MASK] menjalankan pekerjaannya.", "baseline": "Sebagai pekerja, seorang [MASK] menjalankan pekerjaannya.", "status": "kandidat tambahan", "syntax_pattern": "keterangan peran + subjek"},
]
template_rows, example_rows = [], []
for spec in template_specs:
    example_rows.extend({"template_id": spec["template_id"], "occupation": row.occupation_text, "kalimat": spec["job"].format(occupation=row.occupation_text)} for row in df_analysis.head(3).itertuples(index=False))
    for kind in ["job", "baseline"]:
        rendered = spec[kind].format(occupation=df_analysis.iloc[0]["occupation_text"])
        length = len(tokenizer(rendered, add_special_tokens=True, truncation=False)["input_ids"])
        template_rows.append({"template_id": spec["template_id"], "kind": kind, "mask_count": rendered.count(tokenizer.mask_token), "token_length": length, "within_model_limit": length <= MAX_SEQUENCE_LENGTH, "occupation_inserted": kind == "baseline" or "{occupation}" in spec[kind], "status": "valid" if rendered.strip() and rendered.count(tokenizer.mask_token) == 1 and length <= MAX_SEQUENCE_LENGTH else "invalid"})
template_validation = pd.DataFrame(template_rows)
display(template_validation)
display(pd.DataFrame(example_rows))
assert (template_validation["status"] == "valid").all()


## 3A. Katalog delapan template

T1–T4 dipertahankan sebagai template lama. T5–T8 adalah kandidat tambahan yang dibuat netral secara operasional dan tetap memerlukan pemeriksaan validator manusia.


In [ ]:
template_catalog_rows = []
for spec in template_specs:
    job_example = spec["job"].format(occupation=df_analysis.iloc[0]["occupation_text"])
    base_example = spec["baseline"].format(occupation=df_analysis.iloc[0]["occupation_text"])
    template_catalog_rows.append({"template_id": spec["template_id"], "template_text": spec["job"], "baseline_text": spec["baseline"], "mask_position": job_example.index("[MASK]"), "syntax_pattern": spec["syntax_pattern"], "job_token_count": len(tokenizer(job_example)["input_ids"]), "baseline_token_count": len(tokenizer(base_example)["input_ids"]), "template_status": spec["status"], "manual_review_note": "Belum tervalidasi linguistik oleh manusia.", "programmatic_status": "valid" if job_example.count("[MASK]") == 1 and len(tokenizer(job_example)["input_ids"]) <= MAX_SEQUENCE_LENGTH else "invalid"})
template_catalog = pd.DataFrame(template_catalog_rows)
display(template_catalog)
assert len(template_catalog) == 8 and (template_catalog["programmatic_status"] == "valid").all()


## 7. Perhitungan LPBS

Probabilitas dan log-probability mentah disimpan untuk jabatan dan baseline. Probabilitas kandidat multi-token dilaporkan sebagai exp dari log-probability rata-rata per subtoken; jumlah log-probability mentah juga dipertahankan dalam perhitungan internal.


In [ ]:
def context_parts(template_text, occupation=None):
    rendered = template_text.format(occupation=occupation) if occupation is not None else template_text
    assert rendered.count("[MASK]") == 1
    return rendered.split("[MASK]")

def score_contexts(contexts, terms):
    ids_by_term = {term: tokenizer(term, add_special_tokens=False)["input_ids"] for term in terms}
    singles = [term for term in terms if len(ids_by_term[term]) == 1]
    multis = [term for term in terms if len(ids_by_term[term]) > 1]
    result = {term: {"raw_logprob": np.full(len(contexts), np.nan), "logprob": np.full(len(contexts), np.nan), "probability": np.full(len(contexts), np.nan), "token_count": len(ids_by_term[term])} for term in terms}
    single_ids = []
    for prefix, suffix in contexts:
        pids = tokenizer(prefix, add_special_tokens=False)["input_ids"]
        sids = tokenizer(suffix, add_special_tokens=False)["input_ids"]
        ids = tokenizer.build_inputs_with_special_tokens(pids + [tokenizer.mask_token_id] + sids)
        assert len(ids) <= MAX_SEQUENCE_LENGTH
        single_ids.append(ids)
    for start in tqdm(range(0, len(single_ids), INFERENCE_BATCH_SIZE), desc="single-token contexts"):
        batch_ids = single_ids[start:start + INFERENCE_BATCH_SIZE]
        batch = tokenizer.pad({"input_ids": batch_ids}, return_tensors="pt").to(device)
        with torch.no_grad():
            log_probs = torch.log_softmax(model(**batch).logits, dim=-1)
        for local, ids in enumerate(batch_ids):
            position = ids.index(tokenizer.mask_token_id)
            values = log_probs[local, position]
            for term in singles:
                raw = float(values[ids_by_term[term][0]].item())
                index = start + local
                result[term]["raw_logprob"][index] = raw
                result[term]["logprob"][index] = raw
                result[term]["probability"][index] = float(np.clip(math.exp(raw), np.finfo(float).tiny, 1.0))
    for term in multis:
        candidate = ids_by_term[term]
        tasks = []
        for index, (prefix, suffix) in enumerate(contexts):
            pids = tokenizer(prefix, add_special_tokens=False)["input_ids"]
            sids = tokenizer(suffix, add_special_tokens=False)["input_ids"]
            for position, token_id in enumerate(candidate):
                candidate_masked = candidate.copy()
                candidate_masked[position] = tokenizer.mask_token_id
                ids = tokenizer.build_inputs_with_special_tokens(pids + candidate_masked + sids)
                assert len(ids) <= MAX_SEQUENCE_LENGTH
                tasks.append((index, token_id, ids))
        raw_sums = np.zeros(len(contexts))
        for start in tqdm(range(0, len(tasks), INFERENCE_BATCH_SIZE), desc=f"PLL {term}"):
            task_batch = tasks[start:start + INFERENCE_BATCH_SIZE]
            batch_ids = [task[2] for task in task_batch]
            batch = tokenizer.pad({"input_ids": batch_ids}, return_tensors="pt").to(device)
            with torch.no_grad():
                log_probs = torch.log_softmax(model(**batch).logits, dim=-1)
            for local, (_, token_id, ids) in enumerate(task_batch):
                position = ids.index(tokenizer.mask_token_id)
                raw_sums[task_batch[local][0]] += float(log_probs[local, position, token_id].item())
        normalized = raw_sums / len(candidate)
        result[term]["raw_logprob"] = raw_sums
        result[term]["logprob"] = normalized
        result[term]["probability"] = np.clip(np.exp(normalized), np.finfo(float).tiny, 1.0)
    return result

score_terms = ["pria", "wanita", "laki-laki", "perempuan"]
job_contexts, context_keys = [], []
for row in df_analysis.itertuples(index=False):
    for spec in template_specs:
        prefix, suffix = context_parts(spec["job"], row.occupation_text)
        job_contexts.append((prefix, suffix))
        context_keys.append((row.occupation_code, spec["template_id"]))
job_scores = score_contexts(job_contexts, score_terms)
baseline_scores = {}
for spec in template_specs:
    prefix, suffix = context_parts(spec["baseline"])
    baseline_scores[spec["template_id"]] = score_contexts([(prefix, suffix)], score_terms)


## 2A. Audit tokenisasi gender dan probabilitas dasar

Probabilitas dasar ditampilkan per template untuk pria, wanita, laki-laki, dan perempuan. Kondisi utama tetap pria–wanita; pasangan laki-laki–perempuan hanya sensitivitas.


In [ ]:
gender_tokenization_audit = tokenization_audit.copy()
for template_id, scores in baseline_scores.items():
    for term in score_terms:
        mask = gender_tokenization_audit["term"].eq(term)
        gender_tokenization_audit.loc[mask, f"baseline_probability_{template_id}"] = scores[term]["probability"][0]
        gender_tokenization_audit.loc[mask, f"baseline_logprob_{template_id}"] = scores[term]["logprob"][0]
gender_tokenization_audit["comparability_note"] = np.where(gender_tokenization_audit["token_count"].eq(1), "single-token", "normalized PLL multi-token; tidak identik dengan single-token")
display(gender_tokenization_audit)


In [ ]:
long_rows = []
for index, (occupation_code, template_id) in enumerate(context_keys):
    source = df_analysis.loc[df_analysis["occupation_code"].eq(occupation_code)].iloc[0]
    base = baseline_scores[template_id]
    for spec in gender_pair_specs:
        male, female = spec["male_term"], spec["female_term"]
        male_job, female_job = job_scores[male]["logprob"][index], job_scores[female]["logprob"][index]
        male_base, female_base = base[male]["logprob"][0], base[female]["logprob"][0]
        male_change, female_change = male_job - male_base, female_job - female_base
        long_rows.append({"occupation_id": source["occupation_code"], "occupation_code": source["occupation_code"], "occupation_name": source["occupation_name_raw"], "occupation_name_raw": source["occupation_name_raw"], "occupation_text": source["occupation_text"], "kbji_group_code": source["submajor_group_code"], "kbji_group_name": source["submajor_group_name"], "kbji_detail_code": source["group_code"], "kbji_detail_name": source["group_name"], "template_id": template_id, "gender_pair": spec["gender_pair"], "condition": spec["condition"], "male_term": male, "female_term": female, "male_token_count": len(tokenizer.tokenize(male)), "female_token_count": len(tokenizer.tokenize(female)), "male_logprob_job_raw": job_scores[male]["raw_logprob"][index], "male_logprob_job": male_job, "female_logprob_job_raw": job_scores[female]["raw_logprob"][index], "female_logprob_job": female_job, "male_logprob_baseline_raw": base[male]["raw_logprob"][0], "male_logprob_baseline": male_base, "female_logprob_baseline_raw": base[female]["raw_logprob"][0], "female_logprob_baseline": female_base, "male_probability_job_raw": math.exp(job_scores[male]["raw_logprob"][index]), "male_probability_job": job_scores[male]["probability"][index], "female_probability_job_raw": math.exp(job_scores[female]["raw_logprob"][index]), "female_probability_job": job_scores[female]["probability"][index], "male_probability_baseline_raw": math.exp(base[male]["raw_logprob"][0]), "male_probability_baseline": base[male]["probability"][0], "female_probability_baseline_raw": math.exp(base[female]["raw_logprob"][0]), "female_probability_baseline": base[female]["probability"][0], "male_logprob_change": male_change, "female_logprob_change": female_change, "lpbs_score": male_change - female_change})
lpbs_scores_long = pd.DataFrame(long_rows)
display(lpbs_scores_long.head())


## 4A. Pseudo-log-likelihood dan pseudo-perplexity kalimat

Setiap token non-spesial pada kalimat lengkap dimask bergantian. Jumlah log-probability dibagi jumlah token lalu diubah menjadi pseudo-perplexity. Nilai ini hanya kewajaran relatif menurut IndoBERT, bukan validasi tata bahasa manusia; template tidak dihapus otomatis.


In [ ]:
def pseudo_perplexity_batch(sentences, batch_size=512):
    encoded = [tokenizer(s, add_special_tokens=True, truncation=False)["input_ids"] for s in sentences]
    tasks, counts = [], np.zeros(len(sentences), dtype=int)
    for sentence_index, ids in enumerate(encoded):
        assert len(ids) <= MAX_SEQUENCE_LENGTH
        for position in range(1, len(ids) - 1):
            masked = ids.copy()
            target = masked[position]
            masked[position] = tokenizer.mask_token_id
            tasks.append((sentence_index, position, target, masked))
            counts[sentence_index] += 1
    log_sums = np.zeros(len(sentences))
    for start in tqdm(range(0, len(tasks), batch_size), desc="pseudo-perplexity"):
        batch_tasks = tasks[start:start + batch_size]
        batch_ids = [x[3] for x in batch_tasks]
        batch = tokenizer.pad({"input_ids": batch_ids}, return_tensors="pt").to(device)
        with torch.no_grad():
            log_probs = torch.log_softmax(model(**batch).logits, dim=-1)
        for local, (sentence_index, position, target, _) in enumerate(batch_tasks):
            log_sums[sentence_index] += float(log_probs[local, position, target].item())
    return np.exp(-(log_sums / counts))

job_sentences, job_keys = [], []
for row in df_analysis.itertuples(index=False):
    for spec in template_specs:
        for gender, term in [("male", "pria"), ("female", "wanita")]:
            job_sentences.append(spec["job"].format(occupation=row.occupation_text).replace("[MASK]", term))
            job_keys.append((row.occupation_code, spec["template_id"], gender))
job_pppl = pd.DataFrame(job_keys, columns=["occupation_code", "template_id", "gender"])
job_pppl["pseudo_perplexity_job"] = pseudo_perplexity_batch(job_sentences)
job_pppl = job_pppl.pivot(index=["occupation_code", "template_id"], columns="gender", values="pseudo_perplexity_job").reset_index().rename(columns={"male": "male_pseudo_perplexity_job", "female": "female_pseudo_perplexity_job"})
base_sentences, base_keys = [], []
for spec in template_specs:
    for gender, term in [("male", "pria"), ("female", "wanita")]:
        base_sentences.append(spec["baseline"].replace("[MASK]", term))
        base_keys.append((spec["template_id"], gender))
base_pppl = pd.DataFrame(base_keys, columns=["template_id", "gender"])
base_pppl["pseudo_perplexity_baseline"] = pseudo_perplexity_batch(base_sentences)
base_pppl = base_pppl.pivot(index="template_id", columns="gender", values="pseudo_perplexity_baseline").reset_index().rename(columns={"male": "male_pseudo_perplexity_baseline", "female": "female_pseudo_perplexity_baseline"})
lpbs_scores_long = lpbs_scores_long.merge(job_pppl, on=["occupation_code", "template_id"], how="left", validate="many_to_one").merge(base_pppl, on="template_id", how="left", validate="many_to_one")
lpbs_scores_long["job_pair_pseudo_perplexity"] = lpbs_scores_long[["male_pseudo_perplexity_job", "female_pseudo_perplexity_job"]].mean(axis=1)
lpbs_scores_long["baseline_pair_pseudo_perplexity"] = lpbs_scores_long[["male_pseudo_perplexity_baseline", "female_pseudo_perplexity_baseline"]].mean(axis=1)
display(lpbs_scores_long[["occupation_code", "template_id", "gender_pair", "lpbs_score", "job_pair_pseudo_perplexity"]].head())
assert np.isfinite(lpbs_scores_long[["job_pair_pseudo_perplexity", "baseline_pair_pseudo_perplexity"]]).all().all()


In [ ]:
main_long = lpbs_scores_long[lpbs_scores_long["condition"].eq("utama")].copy()
main_long["direction"] = np.select([main_long["lpbs_score"] > 0, main_long["lpbs_score"] < 0], ["positif", "negatif"], default="nol")
template_scores = main_long.pivot(index="occupation_code", columns="template_id", values="lpbs_score")
robust_occupation_summary = main_long.groupby(["occupation_code", "occupation_name_raw", "occupation_text", "kbji_group_code", "kbji_group_name", "kbji_detail_code", "kbji_detail_name"], as_index=False).agg(valid_template_count=("lpbs_score", "count"), mean_lpbs=("lpbs_score", "mean"), median_lpbs=("lpbs_score", "median"), std_lpbs=("lpbs_score", "std"), minimum_lpbs=("lpbs_score", "min"), maximum_lpbs=("lpbs_score", "max"), positive_template_count=("lpbs_score", lambda x: int((x > 0).sum())), negative_template_count=("lpbs_score", lambda x: int((x < 0).sum())), positive_proportion=("lpbs_score", lambda x: float((x > 0).mean())), negative_proportion=("lpbs_score", lambda x: float((x < 0).mean())))
quartiles = main_long.groupby("occupation_code")["lpbs_score"].quantile([.25, .75]).unstack()
robust_occupation_summary["iqr_lpbs"] = robust_occupation_summary["occupation_code"].map(quartiles[.75] - quartiles[.25])
robust_occupation_summary["mad_lpbs"] = robust_occupation_summary["occupation_code"].map(main_long.groupby("occupation_code")["lpbs_score"].apply(lambda x: float((x - x.median()).abs().mean())))
robust_occupation_summary["range_lpbs"] = robust_occupation_summary["maximum_lpbs"] - robust_occupation_summary["minimum_lpbs"]
robust_occupation_summary["sign_agreement"] = robust_occupation_summary["occupation_code"].map(template_scores.apply(lambda x: max((x > 0).sum(), (x < 0).sum(), (x == 0).sum()) / len(x), axis=1))
robust_occupation_summary["sign_flip_count"] = robust_occupation_summary["occupation_code"].map(template_scores.apply(lambda x: int((np.sign(x) != np.sign(x.median())).sum()), axis=1))
robust_occupation_summary["direction_descriptive"] = np.select([robust_occupation_summary["median_lpbs"] > 0, robust_occupation_summary["median_lpbs"] < 0], ["asosiasi relatif ke pria", "asosiasi relatif ke wanita"], default="hasil tidak konsisten")
robust_occupation_summary["template_variation"] = robust_occupation_summary["range_lpbs"]
display(robust_occupation_summary.head())
display(robust_occupation_summary["direction_descriptive"].value_counts().rename("jumlah_jabatan").to_frame())
display(robust_occupation_summary.nlargest(20, "median_lpbs")[["occupation_code", "occupation_name_raw", "median_lpbs", "sign_agreement"]])
display(robust_occupation_summary.nsmallest(20, "median_lpbs")[["occupation_code", "occupation_name_raw", "median_lpbs", "sign_agreement"]])
display(robust_occupation_summary.nlargest(20, "template_variation")[["occupation_code", "occupation_name_raw", "template_variation", "median_lpbs"]])
assert len(robust_occupation_summary) == 415 and (robust_occupation_summary["valid_template_count"] == len(template_specs)).all()


LPBS positif berarti informasi jabatan meningkatkan perbedaan relatif ke istilah laki-laki; LPBS negatif berarti relatif ke istilah perempuan. Skor ini bukan predikat bahwa jabatan tersebut cocok untuk gender tertentu.


In [ ]:
main_long = lpbs_scores_long[lpbs_scores_long["condition"].eq("utama")].copy()
main_long["direction"] = np.select([main_long["lpbs_score"] > 0, main_long["lpbs_score"] < 0], ["positif", "negatif"], default="nol")
qc_intermediate = pd.DataFrame([
    {"check": "jumlah baris utama", "status": len(main_long) == len(df_analysis) * len(template_specs), "observed": len(main_long)},
    {"check": "kelompok 21-26", "status": set(main_long["kbji_group_code"]) == allowed_groups, "observed": sorted(main_long["kbji_group_code"].unique())},
    {"check": "angka finite", "status": bool(np.isfinite(lpbs_scores_long.select_dtypes(include=np.number).to_numpy()).all()), "observed": int(lpbs_scores_long.isna().sum().sum())},
])
display(qc_intermediate)
assert qc_intermediate["status"].all()


## 9. Agregasi skor per jabatan

Median lintas-template menjadi agregat utama. Rata-rata, median, simpangan baku, IQR, rentang, proporsi tanda, dan agreement tanda tetap disimpan agar variasi konteks dapat diaudit.


In [ ]:
group_keys = ["occupation_code", "occupation_name_raw", "occupation_text", "kbji_group_code", "kbji_group_name", "kbji_detail_code", "kbji_detail_name"]
occupation_summary = main_long.groupby(group_keys, as_index=False).agg(valid_template_count=("lpbs_score", "count"), mean_lpbs=("lpbs_score", "mean"), median_lpbs=("lpbs_score", "median"), std_lpbs=("lpbs_score", "std"), minimum_lpbs=("lpbs_score", "min"), maximum_lpbs=("lpbs_score", "max"), positive_proportion=("lpbs_score", lambda x: (x > 0).mean()), negative_proportion=("lpbs_score", lambda x: (x < 0).mean()))
quartiles = main_long.groupby("occupation_code")["lpbs_score"].quantile([.25, .75]).unstack()
occupation_summary["iqr_lpbs"] = occupation_summary["occupation_code"].map(quartiles[.75] - quartiles[.25])
sign_matrix = main_long.pivot(index="occupation_code", columns="template_id", values="lpbs_score")
sign_stats = pd.DataFrame({
    "occupation_code": sign_matrix.index,
    "sign_agreement": sign_matrix.apply(lambda x: max((x > 0).sum(), (x < 0).sum(), (x == 0).sum()) / len(x), axis=1).to_numpy(),
    "sign_change_count": sign_matrix.apply(lambda x: int((np.sign(x) != np.sign(x.median())).sum()), axis=1).to_numpy(),
})
occupation_summary = occupation_summary.merge(sign_stats, on="occupation_code", how="left", validate="one_to_one")
occupation_summary["score_range"] = occupation_summary["maximum_lpbs"] - occupation_summary["minimum_lpbs"]
occupation_summary["direction_descriptive"] = np.select([occupation_summary["median_lpbs"] > 0, occupation_summary["median_lpbs"] < 0], ["positif", "negatif"], default="nol")
display(occupation_summary.head())
display(occupation_summary["direction_descriptive"].value_counts().rename("jumlah_jabatan").to_frame())


## 10. Enam kelompok KBJI dan cluster bootstrap

Kode 21–26 adalah unit utama. Interval percentile 95% memakai median dan 5.000 pengulangan. Jika kode tiga digit tersedia, baris di-resampling berdasarkan cluster kode tersebut.


In [ ]:
group_rows = []
rng_bootstrap = np.random.default_rng(RANDOM_SEED)
for group in sorted(allowed_groups):
    subset = occupation_summary[occupation_summary["kbji_group_code"].astype(str).eq(group)].copy()
    clusters = subset["kbji_detail_code"].dropna().astype(str).unique()
    values = []
    for _ in range(BOOTSTRAP_REPETITIONS):
        sampled = pd.concat([subset[subset["kbji_detail_code"].astype(str).eq(c)] for c in rng_bootstrap.choice(clusters, len(clusters), replace=True)], ignore_index=True)
        values.append(float(sampled["median_lpbs"].median()))
    group_rows.append({"kbji_group_code": group, "kbji_group_name": subset["kbji_group_name"].iloc[0], "n_occupation": len(subset), "n_cluster": len(clusters), "cluster_unit": "group_code tiga digit", "mean_lpbs": subset["median_lpbs"].mean(), "median_lpbs": subset["median_lpbs"].median(), "std_lpbs": subset["median_lpbs"].std(), "iqr_lpbs": subset["median_lpbs"].quantile(.75) - subset["median_lpbs"].quantile(.25), "minimum_lpbs": subset["median_lpbs"].min(), "maximum_lpbs": subset["median_lpbs"].max(), "positive_proportion": (subset["median_lpbs"] > 0).mean(), "negative_proportion": (subset["median_lpbs"] < 0).mean(), "ci95_lower": np.percentile(values, 2.5), "ci95_upper": np.percentile(values, 97.5)})
group_summary = pd.DataFrame(group_rows)
display(group_summary)
display(Markdown("Interval yang melintasi nol belum menetapkan arah kelompok; interval yang seluruhnya di atas atau di bawah nol menunjukkan kecenderungan relatif yang lebih konsisten."))


## 11. Perbandingan antarkelompok

Kruskal–Wallis dan epsilon-squared adalah analisis inferensial eksploratif. Perbandingan pasangan hanya dilanjutkan bila omnibus memberi alasan eksploratif dan dikoreksi dengan Holm. Hierarki KBJI dapat membuat observasi tidak independen penuh.


In [ ]:
arrays = [occupation_summary.loc[occupation_summary["kbji_group_code"].astype(str).eq(g), "median_lpbs"].to_numpy() for g in sorted(allowed_groups)]
kw = kruskal(*arrays)
epsilon_squared = max(0.0, (float(kw.statistic) - len(arrays) + 1) / (len(occupation_summary) - len(arrays)))
statistical_rows = [{"test": "Kruskal-Wallis", "comparison": "21 vs 22 vs 23 vs 24 vs 25 vs 26", "statistic": float(kw.statistic), "p_value": float(kw.pvalue), "adjusted_p_value": np.nan, "effect_size": epsilon_squared, "interpretation": "eksploratif"}]
if kw.pvalue < 0.05:
    pair_rows = []
    groups_sorted = sorted(allowed_groups)
    for i, a in enumerate(groups_sorted):
        for b in groups_sorted[i + 1:]:
            result = mannwhitneyu(occupation_summary.loc[occupation_summary["kbji_group_code"].astype(str).eq(a), "median_lpbs"], occupation_summary.loc[occupation_summary["kbji_group_code"].astype(str).eq(b), "median_lpbs"], alternative="two-sided")
            pair_rows.append({"test": "Mann-Whitney U", "comparison": f"{a} vs {b}", "statistic": float(result.statistic), "p_value": float(result.pvalue), "adjusted_p_value": np.nan, "effect_size": np.nan, "interpretation": "eksploratif; Holm"})
    for row, adj in zip(pair_rows, multipletests([x["p_value"] for x in pair_rows], method="holm")[1]):
        row["adjusted_p_value"] = float(adj)
    statistical_rows.extend(pair_rows)
statistical_tests = pd.DataFrame(statistical_rows)
display(statistical_tests)


## 12. Sensitivitas template

Dihitung median per template, korelasi Spearman, agreement tanda, jumlah sign flip, dan leave-one-template-out pada tingkat kelompok. Template tidak dibuang berdasarkan korelasi atau hasil uji.


In [ ]:
template_summary = main_long.groupby("template_id", as_index=False).agg(n_occupation=("occupation_code", "count"), median_lpbs=("lpbs_score", "median"), mean_lpbs=("lpbs_score", "mean"), positive_proportion=("lpbs_score", lambda x: (x > 0).mean()), negative_proportion=("lpbs_score", lambda x: (x < 0).mean()))
template_stability_rows = []
for i, a in enumerate(sign_matrix.columns):
    for b in sign_matrix.columns[i + 1:]:
        pair = sign_matrix[[a, b]].dropna()
        template_stability_rows.append({"analysis": "pairwise_template", "template_a": a, "template_b": b, "kbji_group_code": "semua", "spearman": spearmanr(pair[a], pair[b]).statistic, "sign_agreement": float((np.sign(pair[a]) == np.sign(pair[b])).mean()), "sign_flip_count": int((np.sign(pair[a]) != np.sign(pair[b])).sum()), "median_absolute_difference": float((pair[a] - pair[b]).abs().median())})
for omitted in sign_matrix.columns:
    kept = sign_matrix.drop(columns=omitted).median(axis=1)
    for group in sorted(allowed_groups):
        codes = occupation_summary.loc[occupation_summary["kbji_group_code"].astype(str).eq(group), "occupation_code"]
        template_stability_rows.append({"analysis": "leave_one_template_out", "template_a": omitted, "template_b": "median_kept", "kbji_group_code": group, "spearman": np.nan, "sign_agreement": np.nan, "sign_flip_count": np.nan, "median_absolute_difference": float((kept.loc[codes].median() - occupation_summary.set_index("occupation_code").loc[codes, "median_lpbs"].median()))})
template_stability = pd.DataFrame(template_stability_rows)
display(template_summary)
display(template_stability)


## 13. Sensitivitas istilah gender

Kondisi utama pria–wanita dibandingkan dengan laki-laki–perempuan. Perbedaan hasil dibaca sebagai sensitivitas tokenisasi, frekuensi, register, penggunaan bahasa, dan korpus pelatihan, bukan sebagai dasar memilih pasangan paling signifikan.


In [ ]:
lexical_wide = lpbs_scores_long.pivot_table(index=["occupation_code", "occupation_name_raw", "kbji_group_code", "kbji_group_name"], columns="condition", values="lpbs_score", aggfunc="median").reset_index()
lexical_wide["difference_sensitivity_minus_main"] = lexical_wide["sensitivitas leksikal"] - lexical_wide["utama"]
lexical_wide["absolute_difference"] = lexical_wide["difference_sensitivity_minus_main"].abs()
lexical_wide["same_direction"] = np.sign(lexical_wide["utama"]) == np.sign(lexical_wide["sensitivitas leksikal"])
lexical_group = lexical_wide.groupby("kbji_group_code", as_index=False).agg(main_median=("utama", "median"), sensitivity_median=("sensitivitas leksikal", "median"), median_absolute_difference=("absolute_difference", "median"), sign_agreement=("same_direction", "mean"), sign_flip_count=("same_direction", lambda x: int((~x).sum())))
lexical_stats = pd.DataFrame([{"metric": "Spearman skor jabatan", "value": spearmanr(lexical_wide["utama"], lexical_wide["sensitivitas leksikal"]).statistic}, {"metric": "perbedaan median", "value": lexical_wide["sensitivitas leksikal"].median() - lexical_wide["utama"].median()}, {"metric": "median absolute difference", "value": lexical_wide["absolute_difference"].median()}, {"metric": "sign agreement", "value": lexical_wide["same_direction"].mean()}, {"metric": "sign flip count", "value": int((~lexical_wide["same_direction"]).sum())}])
lexical_sensitivity = pd.concat([lexical_wide.assign(analysis="occupation"), lexical_group.assign(analysis="group"), lexical_stats.assign(analysis="overall")], ignore_index=True, sort=False)
display(lexical_stats)
display(lexical_group)


## 14. Perbandingan dengan baseline fastText/SC-WEAT

Baseline dibaca dari output fastText yang sudah ada jika kolomnya kompatibel. LPBS dan SC-WEAT berbeda skala; perbandingan dibatasi pada arah, peringkat, korelasi, dan agreement.


In [ ]:
fasttext_path = ROOT / "outputs" / "test_v2_hasil_scweat.csv"
fasttext_comparison = pd.DataFrame()
fasttext_group_comparison = pd.DataFrame()
fasttext_message = "Baseline fastText dilewati: file tidak ditemukan."
if fasttext_path.exists():
    fasttext = pd.read_csv(fasttext_path, dtype={"occupation_code": str})
    if {"occupation_code", "scweat_effect_size"}.issubset(fasttext.columns):
        fasttext_comparison = occupation_summary[["occupation_code", "occupation_name_raw", "kbji_group_code", "kbji_group_name", "median_lpbs"]].merge(fasttext[["occupation_code", "scweat_effect_size"]], on="occupation_code", how="inner", validate="one_to_one").rename(columns={"scweat_effect_size": "fasttext_scweat_effect_size"})
        fasttext_comparison["same_direction"] = np.sign(fasttext_comparison["median_lpbs"]) == np.sign(fasttext_comparison["fasttext_scweat_effect_size"])
        fasttext_group_comparison = fasttext_comparison.groupby(["kbji_group_code", "kbji_group_name"], as_index=False).agg(n_occupation=("occupation_code", "count"), indobert_median=("median_lpbs", "median"), fasttext_median=("fasttext_scweat_effect_size", "median"), sign_agreement=("same_direction", "mean"), sign_flip_count=("same_direction", lambda x: int((~x).sum())))
        fasttext_message = f"Baseline fastText tersedia untuk {len(fasttext_comparison)} jabatan."
    else:
        fasttext_message = "Baseline fastText dilewati: kolom kompatibel tidak lengkap."
print(fasttext_message)
if len(fasttext_comparison):
    display(fasttext_comparison.head())
    display(fasttext_group_comparison)
    display(pd.DataFrame([{"metric": "Spearman", "value": spearmanr(fasttext_comparison["median_lpbs"], fasttext_comparison["fasttext_scweat_effect_size"]).statistic}, {"metric": "sign agreement", "value": fasttext_comparison["same_direction"].mean()}, {"metric": "sign flip count", "value": int((~fasttext_comparison["same_direction"]).sum())}]))


## 15. Visualisasi

Semua grafik ditampilkan langsung. Jabatan dengan skor ekstrem hanya bersifat deskriptif dan harus dibaca bersama kestabilan template.


In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(8, 5)); sns.histplot(df_all["observed_token_count"], discrete=True, color="#4C78A8"); plt.title("Distribusi panjang nama jabatan"); plt.xlabel("Jumlah token"); plt.ylabel("Jumlah jabatan"); plt.show()
plt.figure(figsize=(8, 5)); sns.histplot(occupation_summary["median_lpbs"], kde=True, color="#F58518"); plt.axvline(0, color="black"); plt.title("Distribusi skor LPBS seluruh jabatan"); plt.xlabel("Median LPBS"); plt.ylabel("Jumlah jabatan"); plt.show()
plt.figure(figsize=(10, 5)); sns.violinplot(data=occupation_summary, x="kbji_group_code", y="median_lpbs", inner="box", color="#72B7B2"); plt.axhline(0, color="black"); plt.title("Distribusi LPBS menurut kelompok 21–26"); plt.xlabel("Kode kelompok"); plt.ylabel("Median LPBS"); plt.show()
plt.figure(figsize=(9, 5)); plt.errorbar(group_summary["kbji_group_code"], group_summary["median_lpbs"], yerr=[group_summary["median_lpbs"] - group_summary["ci95_lower"], group_summary["ci95_upper"] - group_summary["median_lpbs"]], fmt="o", capsize=4); plt.axhline(0, color="black"); plt.title("Median kelompok dengan interval kepercayaan 95%"); plt.xlabel("Kode kelompok"); plt.ylabel("Median LPBS"); plt.show()
heat = main_long.groupby(["kbji_group_code", "template_id"])["lpbs_score"].median().unstack()
plt.figure(figsize=(8, 5)); sns.heatmap(heat, center=0, cmap="RdBu_r", annot=True, fmt=".3f"); plt.title("Median skor kelompok × template"); plt.xlabel("Template"); plt.ylabel("Kode kelompok"); plt.show()
plt.figure(figsize=(7, 6)); sns.heatmap(sign_matrix.corr(method="spearman"), vmin=-1, vmax=1, center=0, cmap="RdBu_r", annot=True, fmt=".2f"); plt.title("Korelasi Spearman antartemplate"); plt.show()
group_positive = main_long.assign(positive=main_long["lpbs_score"] > 0).groupby("kbji_group_code")["positive"].mean().reset_index(name="proporsi_positif")
plt.figure(figsize=(8, 5)); sns.barplot(data=group_positive, x="kbji_group_code", y="proporsi_positif", color="#59A14F"); plt.axhline(.5, color="black", linestyle="--"); plt.title("Sign agreement positif per kelompok"); plt.xlabel("Kode kelompok"); plt.ylabel("Proporsi positif"); plt.show()
flip_plot = template_stability[template_stability["analysis"].eq("pairwise_template")]
plt.figure(figsize=(9, 5)); sns.barplot(data=flip_plot, x="template_a", y="sign_flip_count", hue="template_b"); plt.title("Jumlah sign flip akibat perubahan template"); plt.xlabel("Template A"); plt.ylabel("Jumlah perubahan tanda"); plt.show()
display(occupation_summary.nlargest(15, "median_lpbs")[["occupation_code", "occupation_name_raw", "kbji_group_code", "median_lpbs", "sign_agreement"]])
display(occupation_summary.nsmallest(15, "median_lpbs")[["occupation_code", "occupation_name_raw", "kbji_group_code", "median_lpbs", "sign_agreement"]])
if len(fasttext_comparison):
    plt.figure(figsize=(7, 7)); sns.scatterplot(data=fasttext_comparison, x="fasttext_scweat_effect_size", y="median_lpbs", hue="kbji_group_code"); plt.axhline(0, color="black"); plt.axvline(0, color="black"); plt.title("Perbandingan fastText dan IndoBERT"); plt.xlabel("Effect size SC-WEAT fastText"); plt.ylabel("Median LPBS IndoBERT"); plt.show()


## 16. Interpretasi aktual dan kesimpulan

Teks berikut dibuat dari hasil yang baru dihitung. Interval kepercayaan dan sensitivitas menjadi bagian dari interpretasi, bukan hanya angka omnibus.


In [ ]:
positive_share = float((occupation_summary["median_lpbs"] > 0).mean())
negative_share = float((occupation_summary["median_lpbs"] < 0).mean())
template_agreement = float(template_stability.loc[template_stability["analysis"].eq("pairwise_template"), "sign_agreement"].median())
lexical_rho = float(lexical_stats.loc[lexical_stats["metric"].eq("Spearman skor jabatan"), "value"].iloc[0])
group_text = "; ".join(f"{r.kbji_group_code}: median {r.median_lpbs:.4f}, CI95% [{r.ci95_lower:.4f}, {r.ci95_upper:.4f}]" for r in group_summary.itertuples())
display(Markdown(f"**Ringkasan aktual:** {len(occupation_summary)} jabatan dianalisis; proporsi median LPBS positif {positive_share:.3f} dan negatif {negative_share:.3f}. Kelompok: {group_text}. Median agreement tanda antartemplate {template_agreement:.3f}; korelasi Spearman kondisi utama dan sensitivitas istilah {lexical_rho:.3f}. Hasil ini menjawab distribusi asosiasi model dan perbedaan deskriptif kode 21–26, tetapi bukan fakta sosial. Stabilitas yang tidak sempurna menunjukkan ketergantungan pada formulasi kalimat atau istilah. Keterbatasan utama adalah satu model, template buatan, lexical set operasional, dependensi hierarkis, dan jarak antara intrinsic association dengan downstream behavior."))


## Referensi metodologis

- Wilie et al. (2020), IndoNLU: https://aclanthology.org/2020.aacl-main.85/
- Model IndoBERT: https://huggingface.co/indobenchmark/indobert-base-p1
- Kurita et al. (2019), probability-based bias measurement: https://aclanthology.org/W19-3823/
- Husse dan Spitz (2022), sensitivitas pengukuran bias model kontekstual: https://aclanthology.org/2022.findings-emnlp.311/
- Goldfarb-Tarrant et al. (2021), batas intrinsic dan downstream bias: https://aclanthology.org/2021.acl-long.150/


## 17. Menyimpan output dan quality control final

Cell terakhir menyimpan semua tabel dan metadata, memeriksa file input, serta melakukan assertion. Tidak ada gambar yang disimpan.


In [ ]:
print("Penyimpanan output v5 dilakukan pada cell final.")

In [ ]:
full_scores = robust_occupation_summary.set_index("occupation_code")["median_lpbs"]
loo_rows = []
for omitted in template_scores.columns:
    loo = template_scores.drop(columns=omitted).median(axis=1)
    aligned = pd.DataFrame({"full": full_scores, "loo": loo}).dropna()
    loo_rows.append({"omitted_template": omitted, "spearman": spearmanr(aligned["full"], aligned["loo"]).statistic, "kendall": kendalltau(aligned["full"], aligned["loo"]).statistic, "sign_agreement": float((np.sign(aligned["full"]) == np.sign(aligned["loo"])).mean()), "sign_flip_count": int((np.sign(aligned["full"]) != np.sign(aligned["loo"])).sum()), "mean_absolute_difference": float((aligned["full"] - aligned["loo"]).abs().mean()), "median_absolute_difference": float((aligned["full"] - aligned["loo"]).abs().median()), "mean_absolute_rank_change": float((aligned["full"].rank() - aligned["loo"].rank()).abs().mean())})
leave_one_template_out = pd.DataFrame(loo_rows)
display(leave_one_template_out)


In [ ]:
pppl_job_pairs = job_pppl.copy()
pppl_job_pairs["job_pair_pseudo_perplexity"] = pppl_job_pairs[["male_pseudo_perplexity_job", "female_pseudo_perplexity_job"]].mean(axis=1)
pppl_base_pairs = base_pppl.copy()
pppl_base_pairs["baseline_pair_pseudo_perplexity"] = pppl_base_pairs[["male_pseudo_perplexity_baseline", "female_pseudo_perplexity_baseline"]].mean(axis=1)
main_long = main_long.merge(pppl_job_pairs[["occupation_code", "template_id", "job_pair_pseudo_perplexity"]], on=["occupation_code", "template_id"], how="left", validate="one_to_one").merge(pppl_base_pairs[["template_id", "baseline_pair_pseudo_perplexity"]], on="template_id", how="left", validate="many_to_one")
main_long["occupation_median_lpbs"] = main_long["occupation_code"].map(full_scores)
main_long["absolute_lpbs"] = main_long["lpbs_score"].abs()
main_long["template_absolute_deviation"] = (main_long["lpbs_score"] - main_long["occupation_median_lpbs"]).abs()
pppl_summary = pd.DataFrame([{"metric": "Spearman PPPL vs absolute LPBS", "value": spearmanr(main_long["job_pair_pseudo_perplexity"], main_long["absolute_lpbs"]).statistic}, {"metric": "Spearman PPPL vs template variation", "value": spearmanr(main_long["job_pair_pseudo_perplexity"], main_long["template_absolute_deviation"]).statistic}])
main_long["inverse_pppl_weight"] = 1 / main_long["job_pair_pseudo_perplexity"].clip(lower=np.finfo(float).tiny)
weighted_summary = main_long.groupby("occupation_code").apply(lambda x: pd.Series({"weighted_lpbs": np.average(x["lpbs_score"], weights=x["inverse_pppl_weight"]), "median_lpbs": x["lpbs_score"].median()}), include_groups=False).reset_index()
weighted_summary["difference_weighted_minus_median"] = weighted_summary["weighted_lpbs"] - weighted_summary["median_lpbs"]
weighted_summary["same_direction"] = np.sign(weighted_summary["weighted_lpbs"]) == np.sign(weighted_summary["median_lpbs"])
pppl_summary = pd.concat([pppl_summary, pd.DataFrame([{"metric": "Spearman weighted vs median", "value": spearmanr(weighted_summary["weighted_lpbs"], weighted_summary["median_lpbs"]).statistic}, {"metric": "weighted sign agreement", "value": weighted_summary["same_direction"].mean()}, {"metric": "weighted sign flip count", "value": int((~weighted_summary["same_direction"]).sum())}, {"metric": "weighted median absolute difference", "value": weighted_summary["difference_weighted_minus_median"].abs().median()}])], ignore_index=True)
display(pppl_summary)
display(weighted_summary.head())


In [ ]:
group_extra = robust_occupation_summary.groupby(["kbji_group_code", "kbji_group_name"], as_index=False).agg(mad_lpbs=("median_lpbs", lambda x: float((x - x.median()).abs().mean())), median_sign_agreement=("sign_agreement", "median"), consistent_direction_proportion=("sign_agreement", lambda x: float((x >= .75).mean())))
group_summary_v5 = group_summary.merge(group_extra, on=["kbji_group_code", "kbji_group_name"], how="left")
baseline_group = baseline_occupation.groupby("kbji_group_code", as_index=False)["median_lpbs"].median().rename(columns={"median_lpbs": "baseline_v4_median_lpbs"})
group_summary_v5 = group_summary_v5.merge(baseline_group, on="kbji_group_code", how="left")
group_summary_v5["median_difference_from_v4"] = group_summary_v5["median_lpbs"] - group_summary_v5["baseline_v4_median_lpbs"]
display(group_summary_v5)


In [ ]:
ranked = robust_occupation_summary[["kbji_group_code", "median_lpbs"]].copy()
ranked["rank"] = rankdata(ranked["median_lpbs"])
dunn_rows, n_total = [], len(ranked)
for i, a in enumerate(sorted(allowed_groups)):
    for b in sorted(allowed_groups)[i + 1:]:
        left, right = ranked[ranked["kbji_group_code"].astype(str).eq(a)], ranked[ranked["kbji_group_code"].astype(str).eq(b)]
        se = math.sqrt((n_total * (n_total + 1) / 12) * (1 / len(left) + 1 / len(right)))
        z_value = (left["rank"].mean() - right["rank"].mean()) / se
        dunn_rows.append({"group_a": a, "group_b": b, "z": z_value, "p_value": 2 * norm.sf(abs(z_value))})
posthoc_dunn = pd.DataFrame(dunn_rows)
posthoc_dunn["p_value_bh"] = multipletests(posthoc_dunn["p_value"], method="fdr_bh")[1]
posthoc_dunn["significant_bh_0_05"] = posthoc_dunn["p_value_bh"] < .05
display(posthoc_dunn)


In [ ]:
mixed_effects_rows, mixed_effects_status, mixed_effects_warning = [], "not attempted", ""
try:
    mixed_model = smf.mixedlm("lpbs_score ~ C(kbji_group_code) + C(template_id)", main_long, groups=main_long["occupation_code"])
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        mixed_fit = mixed_model.fit(reml=False, method="lbfgs", maxiter=200, disp=False)
    mixed_effects_status = "converged" if bool(getattr(mixed_fit, "converged", False)) else "not converged"
    mixed_effects_warning = " | ".join(str(w.message) for w in caught)
    for name, coef, se, p_value, low, high in zip(mixed_fit.params.index, mixed_fit.params, mixed_fit.bse, mixed_fit.pvalues, mixed_fit.conf_int()[0], mixed_fit.conf_int()[1]):
        mixed_effects_rows.append({"term": name, "coefficient": coef, "standard_error": se, "p_value": p_value, "ci_lower": low, "ci_upper": high, "variance_component": float(mixed_fit.cov_re.iloc[0, 0]), "status_convergence": mixed_effects_status, "warning": mixed_effects_warning})
except Exception as error:
    mixed_effects_status, mixed_effects_warning = "not estimable", repr(error)
    mixed_effects_rows.append({"term": "model", "coefficient": np.nan, "standard_error": np.nan, "p_value": np.nan, "ci_lower": np.nan, "ci_upper": np.nan, "variance_component": np.nan, "status_convergence": mixed_effects_status, "warning": mixed_effects_warning})
mixed_effects = pd.DataFrame(mixed_effects_rows)
display(mixed_effects)


In [ ]:
if len(fasttext_comparison):
    fasttext_codes = set(fasttext_comparison["occupation_code"].astype(str))
    unmatched_codes = sorted(set(df_analysis["occupation_code"].astype(str)) - fasttext_codes)
else:
    unmatched_codes = sorted(df_analysis["occupation_code"].astype(str))
fasttext_unmatched = df_analysis[df_analysis["occupation_code"].astype(str).isin(unmatched_codes)][["occupation_code", "occupation_name_raw", "submajor_group_code", "group_code"]].copy()
fasttext_unmatched["reason"] = "occupation_code tidak tersedia pada outputs/test_v2_hasil_scweat.csv"
display(fasttext_unmatched)
assert len(fasttext_unmatched) in {0, 14}


In [ ]:
plt.figure(figsize=(8, 5)); sns.heatmap(template_scores.corr(method="spearman"), vmin=-1, vmax=1, center=0, cmap="RdBu_r", annot=True, fmt=".2f"); plt.title("Korelasi Spearman delapan template"); plt.show()
loo_heat = leave_one_template_out.set_index("omitted_template")[["sign_agreement", "spearman", "kendall"]]
plt.figure(figsize=(7, 5)); sns.heatmap(loo_heat, center=.5, cmap="YlGnBu", annot=True, fmt=".2f"); plt.title("Stabilitas leave-one-template-out"); plt.show()
plt.figure(figsize=(8, 5)); sns.barplot(data=leave_one_template_out, x="omitted_template", y="sign_flip_count", color="#F58518"); plt.title("Sign flip setelah satu template dikeluarkan"); plt.show()
plt.figure(figsize=(8, 5)); sns.scatterplot(data=main_long, x="job_pair_pseudo_perplexity", y="absolute_lpbs", hue="template_id"); plt.title("Pseudo-perplexity dan absolute LPBS"); plt.xlabel("Pseudo-perplexity"); plt.ylabel("Absolute LPBS"); plt.show()
plt.figure(figsize=(10, 5)); sns.boxplot(data=robust_occupation_summary, x="kbji_group_code", y="median_lpbs", color="#72B7B2"); plt.axhline(0, color="black"); plt.title("Agregasi robust LPBS per kelompok KBJI"); plt.show()
plt.figure(figsize=(8, 6)); sns.scatterplot(data=lexical_wide, x="utama", y="sensitivitas leksikal", hue="kbji_group_code"); plt.axhline(0, color="black"); plt.axvline(0, color="black"); plt.title("Sensitivitas pasangan istilah gender"); plt.show()


## Output tabel test2_v5

Tabel robustness disimpan setelah seluruh analisis selesai. Visualisasi hanya ditampilkan di notebook.


In [ ]:
outputs_v5 = {"test2_v5_template_catalog.csv": template_catalog, "test2_v5_gender_tokenization_audit.csv": gender_tokenization_audit, "test2_v5_sentence_level_lpbs.csv": lpbs_scores_long, "test2_v5_occupation_robust_summary.csv": robust_occupation_summary.merge(weighted_summary[["occupation_code", "weighted_lpbs", "difference_weighted_minus_median", "same_direction"]], on="occupation_code", how="left"), "test2_v5_group_summary.csv": group_summary_v5, "test2_v5_template_stability.csv": template_stability, "test2_v5_leave_one_template_out.csv": leave_one_template_out, "test2_v5_lexical_sensitivity.csv": lexical_sensitivity, "test2_v5_pppl_summary.csv": pppl_summary, "test2_v5_posthoc_dunn.csv": posthoc_dunn, "test2_v5_mixed_effects.csv": mixed_effects, "test2_v5_fasttext_unmatched.csv": fasttext_unmatched}
for filename, frame in outputs_v5.items():
    frame.to_csv(OUTPUT_DIR / filename, index=False, encoding="utf-8")


## Kesimpulan faktual test2_v5

Kesimpulan berikut dihitung dari hasil aktual dan dibaca sebagai asosiasi model. Pseudo-perplexity, weighted score, Dunn, dan mixed-effects adalah analisis tambahan.


In [ ]:
positive_count = int((robust_occupation_summary["median_lpbs"] > 0).sum())
negative_count = int((robust_occupation_summary["median_lpbs"] < 0).sum())
consistent_count = int((robust_occupation_summary["sign_agreement"] >= .75).sum())
display(Markdown(f"**Ringkasan aktual:** {positive_count} jabatan positif dan {negative_count} negatif pada median LPBS; {consistent_count} memiliki sign agreement minimal 0,75. Leave-one-template-out memiliki rentang Spearman {leave_one_template_out.spearman.min():.3f}–{leave_one_template_out.spearman.max():.3f}. Template dengan sign flip terbanyak: {leave_one_template_out.loc[leave_one_template_out.sign_flip_count.idxmax(), 'omitted_template']}. Weighted inverse PPPL menghasilkan {int((~weighted_summary.same_direction).sum())} sign flip terhadap median. Analisis ini tidak menjadi bukti diskriminasi atau fakta dunia kerja nyata."))


## Quality control final test2_v5

QC terakhir memeriksa data, delapan template, LPBS, pseudo-perplexity, probabilitas, output, dan audit ketidakcocokan baseline.


In [ ]:
required_v5_outputs = list(outputs_v5) + ["test2_v5_quality_control.csv", "test2_v5_run_metadata.json"]
input_hashes_after = {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in [analysis_path, all_path, gender_path]}
quality_rows_v5 = [
    {"check": "model sesuai", "status": MODEL_NAME == "indobenchmark/indobert-base-p1", "observed": MODEL_NAME},
    {"check": "baseline 415 jabatan, 214 positif, 201 negatif", "status": len(baseline_occupation) == 415 and baseline_positive == 214 and baseline_negative == 201, "observed": [len(baseline_occupation), baseline_positive, baseline_negative]},
    {"check": "delapan template valid", "status": len(template_specs) == 8 and (template_catalog["programmatic_status"] == "valid").all(), "observed": len(template_specs)},
    {"check": "baris sentence LPBS sesuai", "status": len(lpbs_scores_long) == len(df_analysis) * len(template_specs) * 2, "observed": len(lpbs_scores_long)},
    {"check": "setiap jabatan memakai delapan template", "status": bool((robust_occupation_summary["valid_template_count"] == 8).all()), "observed": robust_occupation_summary["valid_template_count"].value_counts().to_dict()},
    {"check": "probabilitas finite dan lebih besar dari nol", "status": bool(lpbs_scores_long[[c for c in lpbs_scores_long.columns if "probability" in c]].apply(lambda c: c.between(0, 1).all() and (c > 0).all()).all()), "observed": "checked"},
    {"check": "LPBS dan PPPL finite", "status": bool(np.isfinite(lpbs_scores_long.select_dtypes(include=np.number).to_numpy()).all()), "observed": int(lpbs_scores_long.isna().sum().sum())},
    {"check": "tidak ada duplikat observasi", "status": bool(~lpbs_scores_long.duplicated(["occupation_code", "template_id", "gender_pair"]).any()), "observed": int(lpbs_scores_long.duplicated(["occupation_code", "template_id", "gender_pair"]).sum())},
    {"check": "audit 14 mismatch", "status": len(fasttext_unmatched) in {0, 14}, "observed": len(fasttext_unmatched)},
    {"check": "CSV input tidak berubah", "status": input_hashes_before == input_hashes_after, "observed": input_hashes_after},
    {"check": "semua output terbentuk", "status": all((OUTPUT_DIR / name).exists() for name in required_v5_outputs[:-2]), "observed": [name for name in required_v5_outputs if (OUTPUT_DIR / name).exists()]},
]
quality_control_v5 = pd.DataFrame(quality_rows_v5)
metadata_v5 = {"version": "test2_v5", "notebook": "test2_v5_indobert_lpbs_robustness.ipynb", "model_name": MODEL_NAME, "model_revision": MODEL_REVISION, "tokenizer": tokenizer.__class__.__name__, "python_version": platform.python_version(), "pytorch_version": torch.__version__, "transformers_version": transformers.__version__, "device": str(device), "random_seed": RANDOM_SEED, "n_all_occupations": len(df_all), "n_analysis_occupations": len(df_analysis), "kbji_groups": sorted(allowed_groups), "template_count": len(template_specs), "bootstrap_repetitions": BOOTSTRAP_REPETITIONS, "input_sha256_before": input_hashes_before, "input_sha256_after": input_hashes_after, "mixed_effects_status": mixed_effects_status, "fasttext_unmatched_count": len(fasttext_unmatched), "time_start": TIME_START_ISO, "time_end": time.strftime("%Y-%m-%dT%H:%M:%S%z"), "duration_seconds": time.time() - TIME_START, "quality_control_passed": bool(quality_control_v5["status"].all())}
quality_control_v5.to_csv(OUTPUT_DIR / "test2_v5_quality_control.csv", index=False, encoding="utf-8")
(OUTPUT_DIR / "test2_v5_run_metadata.json").write_text(json.dumps(metadata_v5, ensure_ascii=False, indent=2), encoding="utf-8")
display(quality_control_v5)
display(pd.DataFrame({"output_file": sorted(required_v5_outputs)}))
assert quality_control_v5["status"].all()
print("QC test2_v5 lulus; durasi detik:", round(metadata_v5["duration_seconds"], 2))
